# AI layer: review sentiment
Uses Databricks AI Functions to label review text with sentiment and topic, then summarizes by restaurant. Output tables are in `zomato_ai.gold`.

## Why we classify distinct comments only
The 300,000 reviews contain only 294 distinct comments, so the AI model runs 294 times instead of 300,000.

In [0]:
%sql
SELECT count(*) AS total_reviews,
       count(DISTINCT comment) AS distinct_comments,
       sum(CASE WHEN comment IS NULL THEN 1 ELSE 0 END) AS null_comments
FROM zomato_ai.gold.fact_reviews

total_reviews,distinct_comments,null_comments
300000,294,0


## Building the tables
1. `comment_enrichment`: sentiment and topic for each distinct comment

In [0]:
%sql
CREATE OR REPLACE TABLE zomato_ai.gold.comment_enrichment AS
SELECT
  comment,
  ai_analyze_sentiment(comment) AS sentiment,
  ai_classify(comment, ARRAY(
    'food quality', 'delivery speed', 'packaging',
    'price and value', 'delivery partner behavior', 'other'
  )) AS topic
FROM (SELECT DISTINCT comment FROM zomato_ai.gold.fact_reviews)

num_affected_rows,num_inserted_rows


2. `review_enriched`: joins those labels back to every review

In [0]:
%sql
CREATE OR REPLACE TABLE zomato_ai.gold.review_enriched AS
SELECT r.review_id, r.order_id, r.user_id, r.restaurant_id,
       r.rating, r.comment, r.review_date,
       e.sentiment, e.topic
FROM zomato_ai.gold.fact_reviews r
LEFT JOIN zomato_ai.gold.comment_enrichment e ON r.comment = e.comment

num_affected_rows,num_inserted_rows


## Validation: do the AI labels match the star ratings?
Positive reviews should average about 4.6 stars and negative about 1.5. The groups should total 300,000.

In [0]:
%sql
SELECT sentiment, topic, count(*) AS reviews, round(avg(rating), 2) AS avg_rating
FROM zomato_ai.gold.review_enriched
GROUP BY sentiment, topic
ORDER BY reviews DESC

sentiment,topic,reviews,avg_rating
positive,price and value,51782,4.61
positive,delivery speed,41234,4.62
positive,food quality,39557,4.62
positive,packaging,35585,4.62
neutral,other,33774,3.0
positive,delivery partner behavior,16736,4.61
negative,price and value,16505,1.5
negative,delivery partner behavior,13658,1.5
negative,packaging,12856,1.51
mixed,food quality,11482,3.0


## Per-restaurant sentiment summary
One row per restaurant: review counts by sentiment, average rating, percent negative, and the most common complaint topic. Note: 87 restaurants in the source have no name and are labeled "Unknown restaurant".

In [0]:
%sql
CREATE OR REPLACE TABLE zomato_ai.gold.agg_restaurant_sentiment AS
WITH base AS (
  SELECT
    restaurant_id,
    count(*) AS total_reviews,
    round(avg(rating), 2) AS avg_rating,
    sum(CASE WHEN sentiment = 'positive' THEN 1 ELSE 0 END) AS positive_reviews,
    sum(CASE WHEN sentiment = 'neutral'  THEN 1 ELSE 0 END) AS neutral_reviews,
    sum(CASE WHEN sentiment = 'mixed'    THEN 1 ELSE 0 END) AS mixed_reviews,
    sum(CASE WHEN sentiment = 'negative' THEN 1 ELSE 0 END) AS negative_reviews
  FROM zomato_ai.gold.review_enriched
  GROUP BY restaurant_id
),
complaints AS (
  SELECT
    restaurant_id,
    topic,
    count(*) AS n,
    row_number() OVER (PARTITION BY restaurant_id ORDER BY count(*) DESC, topic) AS rn
  FROM zomato_ai.gold.review_enriched
  WHERE sentiment = 'negative'
  GROUP BY restaurant_id, topic
)
SELECT
  b.restaurant_id,
  coalesce(r.name, 'Unknown restaurant') AS restaurant_name,
  r.city,
  b.total_reviews,
  b.avg_rating,
  b.positive_reviews,
  b.neutral_reviews,
  b.mixed_reviews,
  b.negative_reviews,
  round(b.negative_reviews / b.total_reviews, 4) AS pct_negative,
  c.topic AS top_complaint_topic,
  c.n AS top_complaint_count
FROM base b
LEFT JOIN zomato_ai.gold.dim_restaurants r ON b.restaurant_id = r.restaurant_id
LEFT JOIN complaints c ON b.restaurant_id = c.restaurant_id AND c.rn = 1

num_affected_rows,num_inserted_rows


### Validation: summary totals
Total reviews should equal 300,000 and no restaurant name should be missing.

In [0]:
%sql
SELECT
  count(*) AS restaurants_with_reviews,
  sum(total_reviews) AS total_reviews,
  max(total_reviews) AS most_reviews_for_one_restaurant,
  sum(CASE WHEN total_reviews >= 5 THEN 1 ELSE 0 END) AS restaurants_with_5plus_reviews,
  sum(CASE WHEN restaurant_name IS NULL THEN 1 ELSE 0 END) AS missing_restaurant_name
FROM zomato_ai.gold.agg_restaurant_sentiment

restaurants_with_reviews,total_reviews,most_reviews_for_one_restaurant,restaurants_with_5plus_reviews,missing_restaurant_name
128994,300000,11,8111,72


### restaurants with the highest share of negative reviews
Only restaurants with at least 5 reviews. Most restaurants have very few reviews, so percentages are rough.

In [0]:
%sql
SELECT restaurant_name, city, total_reviews, avg_rating,
       pct_negative, top_complaint_topic
FROM zomato_ai.gold.agg_restaurant_sentiment
WHERE total_reviews >= 5
ORDER BY pct_negative DESC, total_reviews DESC
LIMIT 10

restaurant_name,city,total_reviews,avg_rating,pct_negative,top_complaint_topic
Kathleen,"Central Kolkata,Kolkata",5,1.2,1.0,price and value
Royal Sandwich,"Porur,Chennai",5,1.8,1.0,delivery speed
Shakes & Bakes,"Vastrapur,Ahmedabad",5,1.4,1.0,food quality
GOBRGR@TRYST GOURMET,"Royapettah,Chennai",5,1.6,1.0,delivery partner behavior
London Dairy,"Bibwewadi,Pune",5,1.4,1.0,delivery partner behavior
7TH HEAVEN,"Dwarka,Delhi",6,2.0,0.8333,price and value
Pizza Hunger,Jaunpur,6,2.17,0.8333,price and value
Patz Kitchen,"Kadubeesanahalli,Bangalore",6,2.5,0.8333,delivery partner behavior
Kissan House,"North Mohali,Chandigarh",6,2.0,0.8333,delivery partner behavior
S S BROAST,"Banjara Hills,Hyderabad",6,1.83,0.8333,price and value
